# ADIP Watermark Portal

This notebook keeps the original DWT + DCT watermarking idea, but removes the fixed `original.jpg` / `watermark.jpg` dependency. Run the cells, upload any cover image and any watermark image, then view the embedded result, extracted watermark, image-quality metrics, capacity details, and downloads.

## 1. Install dependencies if needed

Run this cell only if your notebook environment is missing packages.

In [ ]:
# Uncomment and run if needed:
# %pip install opencv-python pywavelets scikit-image pillow matplotlib gradio

## 2. Imports

In [ ]:
import math
import tempfile
from pathlib import Path

import cv2
import gradio as gr
import matplotlib.pyplot as plt
import numpy as np
import pywt
from PIL import Image
from skimage.metrics import structural_similarity as ssim

## 3. Watermarking engine

The original notebook pads to `2 ** level`, but DCT embedding also needs complete `8 x 8` blocks inside the selected DWT band. This version pads to `(2 ** level) * 8`, so arbitrary image sizes work more reliably.

In [ ]:
BLOCK_SIZE = 8
OUTPUT_DIR = Path(tempfile.gettempdir()) / "adip_watermark_portal"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def pil_to_gray(image):
    if image is None:
        raise ValueError("Please upload both a cover image and a watermark image.")
    arr = np.array(image.convert("RGB"))
    return cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)


def binarize_watermark(wm_gray, threshold=127):
    _, binary = cv2.threshold(wm_gray, threshold, 1, cv2.THRESH_BINARY)
    return binary.astype(np.uint8)


def pad_for_embedding(image, level=3, block_size=BLOCK_SIZE):
    multiple = (2 ** level) * block_size
    h, w = image.shape
    new_h = math.ceil(h / multiple) * multiple
    new_w = math.ceil(w / multiple) * multiple
    padded = np.zeros((new_h, new_w), dtype=np.float32)
    padded[:h, :w] = image.astype(np.float32)
    return padded, (h, w), multiple


def watermark_capacity(padded_shape, level=3, block_size=BLOCK_SIZE, wavelet="haar"):
    probe = np.zeros(padded_shape, dtype=np.float32)
    coeffs = pywt.wavedec2(probe, wavelet, level=level)
    cH, _, _ = coeffs[1]
    hl_h, hl_w = cH.shape
    return hl_h // block_size, hl_w // block_size


def prepare_watermark(wm_gray, bit_shape, threshold=127):
    rows, cols = bit_shape
    resized = cv2.resize(wm_gray, (cols, rows), interpolation=cv2.INTER_AREA)
    bits = binarize_watermark(resized, threshold=threshold)
    return resized, bits


def embed_watermark(cover_gray, wm_bits, level=3, alpha=100, wavelet="haar", dct_position=(3, 3)):
    coeffs = pywt.wavedec2(cover_gray.astype(np.float32), wavelet, level=level)
    cA = coeffs[0]
    cH, cV, cD = coeffs[1]
    hl = cH.copy().astype(np.float32)

    rows = hl.shape[0] // BLOCK_SIZE
    cols = hl.shape[1] // BLOCK_SIZE
    if wm_bits.shape != (rows, cols):
        raise ValueError(f"Watermark bit map must be {rows} x {cols}, got {wm_bits.shape}.")

    dy, dx = dct_position
    for by in range(rows):
        for bx in range(cols):
            y0 = by * BLOCK_SIZE
            x0 = bx * BLOCK_SIZE
            block = hl[y0:y0 + BLOCK_SIZE, x0:x0 + BLOCK_SIZE].copy()
            dct_block = cv2.dct(block)
            dct_block[dy, dx] = alpha if wm_bits[by, bx] else -alpha
            hl[y0:y0 + BLOCK_SIZE, x0:x0 + BLOCK_SIZE] = cv2.idct(dct_block)

    coeffs_mod = [cA, (hl, cV, cD)] + coeffs[2:]
    reconstructed = pywt.waverec2(coeffs_mod, wavelet)
    return np.clip(reconstructed, 0, 255).astype(np.uint8)


def extract_watermark(watermarked_gray, level=3, wavelet="haar", dct_position=(3, 3)):
    coeffs = pywt.wavedec2(watermarked_gray.astype(np.float32), wavelet, level=level)
    cH, _, _ = coeffs[1]
    hl = cH.copy().astype(np.float32)
    rows = hl.shape[0] // BLOCK_SIZE
    cols = hl.shape[1] // BLOCK_SIZE
    recovered = np.zeros((rows, cols), dtype=np.uint8)
    dy, dx = dct_position

    for by in range(rows):
        for bx in range(cols):
            y0 = by * BLOCK_SIZE
            x0 = bx * BLOCK_SIZE
            block = hl[y0:y0 + BLOCK_SIZE, x0:x0 + BLOCK_SIZE]
            dct_block = cv2.dct(block)
            recovered[by, bx] = 1 if dct_block[dy, dx] >= 0 else 0

    return (recovered * 255).astype(np.uint8)


def compare_images(original, watermarked):
    original_u8 = np.clip(original, 0, 255).astype(np.uint8)
    watermarked_u8 = np.clip(watermarked, 0, 255).astype(np.uint8)
    mse_value = float(np.mean((original_u8.astype(np.float32) - watermarked_u8.astype(np.float32)) ** 2))
    psnr_value = float(cv2.PSNR(original_u8, watermarked_u8))
    data_range = int(original_u8.max()) - int(original_u8.min()) or 255
    ssim_value = float(ssim(original_u8, watermarked_u8, data_range=data_range))
    return mse_value, psnr_value, ssim_value


def gray_to_pil(image):
    return Image.fromarray(np.clip(image, 0, 255).astype(np.uint8), mode="L")


def save_gray(image, name):
    path = OUTPUT_DIR / name
    gray_to_pil(image).save(path)
    return str(path)

## 4. Portal logic

In [ ]:
def process_uploads(cover_image, watermark_image, level, alpha, threshold, wavelet, dct_y, dct_x):
    cover_gray = pil_to_gray(cover_image)
    watermark_gray = pil_to_gray(watermark_image)

    padded_cover, original_shape, required_multiple = pad_for_embedding(cover_gray, level=level)
    bit_shape = watermark_capacity(padded_cover.shape, level=level, wavelet=wavelet)
    wm_resized, wm_bits = prepare_watermark(watermark_gray, bit_shape, threshold=threshold)

    watermarked_padded = embed_watermark(
        padded_cover,
        wm_bits,
        level=level,
        alpha=alpha,
        wavelet=wavelet,
        dct_position=(int(dct_y), int(dct_x)),
    )
    extracted = extract_watermark(
        watermarked_padded,
        level=level,
        wavelet=wavelet,
        dct_position=(int(dct_y), int(dct_x)),
    )

    h, w = original_shape
    cover_crop = padded_cover[:h, :w]
    watermarked_crop = watermarked_padded[:h, :w]
    mse_value, psnr_value, ssim_value = compare_images(cover_crop, watermarked_crop)

    changed_pixels = int(np.count_nonzero(np.abs(cover_crop.astype(np.int16) - watermarked_crop.astype(np.int16))))
    total_pixels = int(h * w)
    white_bits = int(np.count_nonzero(wm_bits))
    black_bits = int(wm_bits.size - white_bits)

    stats = [
        ["Original cover size", f"{h} x {w}"],
        ["Padded processing size", f"{padded_cover.shape[0]} x {padded_cover.shape[1]}"] ,
        ["Required size multiple", required_multiple],
        ["Watermark bit capacity", f"{bit_shape[0]} x {bit_shape[1]} = {wm_bits.size} bits"],
        ["White watermark bits", white_bits],
        ["Black watermark bits", black_bits],
        ["Changed cover pixels", f"{changed_pixels} / {total_pixels} ({changed_pixels / total_pixels:.2%})"],
        ["MSE", f"{mse_value:.4f}"],
        ["PSNR", f"{psnr_value:.2f} dB"],
        ["SSIM", f"{ssim_value:.4f}"],
        ["DWT level", level],
        ["Alpha strength", alpha],
        ["Wavelet", wavelet],
        ["DCT coefficient", f"({int(dct_y)}, {int(dct_x)})"],
    ]

    gallery = [
        (gray_to_pil(cover_crop), "Cover image"),
        (gray_to_pil(wm_resized), "Watermark resized to capacity"),
        (gray_to_pil(wm_bits * 255), "Binary watermark bits"),
        (gray_to_pil(watermarked_crop), "Watermarked image"),
        (gray_to_pil(extracted), "Extracted watermark"),
    ]

    downloads = [
        save_gray(watermarked_crop, "watermarked_image.png"),
        save_gray(extracted, "extracted_watermark.png"),
        save_gray(wm_bits * 255, "embedded_watermark_bits.png"),
    ]
    return gallery, stats, downloads


def clear_outputs():
    return [], [], []

## 5. Launch the upload portal

After running this cell, use the local URL shown below the cell. In Colab, set `share=True` in `demo.launch()` if you need a public link.

In [ ]:
with gr.Blocks(title="ADIP Watermark Portal", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# ADIP Watermark Portal")
    gr.Markdown("Upload any cover image and watermark image. The portal embeds the watermark, extracts it back, and reports quality and capacity statistics.")

    with gr.Row():
        cover_input = gr.Image(label="Cover image", type="pil", image_mode="RGB")
        watermark_input = gr.Image(label="Watermark image", type="pil", image_mode="RGB")

    with gr.Accordion("Embedding controls", open=True):
        with gr.Row():
            level = gr.Slider(1, 4, value=3, step=1, label="DWT level")
            alpha = gr.Slider(5, 250, value=100, step=5, label="Alpha strength")
            threshold = gr.Slider(0, 255, value=127, step=1, label="Watermark threshold")
        with gr.Row():
            wavelet = gr.Dropdown(["haar", "db1", "db2", "sym2"], value="haar", label="Wavelet")
            dct_y = gr.Slider(1, 6, value=3, step=1, label="DCT row")
            dct_x = gr.Slider(1, 6, value=3, step=1, label="DCT column")

    with gr.Row():
        run_button = gr.Button("Embed and analyze", variant="primary")
        clear_button = gr.Button("Clear")

    gallery = gr.Gallery(label="Results", columns=3, height="auto", object_fit="contain")
    stats = gr.Dataframe(headers=["Statistic", "Value"], datatype=["str", "str"], label="Statistics", wrap=True)
    downloads = gr.Files(label="Download generated files")

    run_button.click(
        process_uploads,
        inputs=[cover_input, watermark_input, level, alpha, threshold, wavelet, dct_y, dct_x],
        outputs=[gallery, stats, downloads],
    )
    clear_button.click(clear_outputs, outputs=[gallery, stats, downloads])

demo.launch()

## Notes for tuning

- Higher `alpha` usually makes extraction stronger but can reduce visual quality.
- Higher DWT levels create a smaller watermark capacity because the HL band is smaller.
- The uploaded watermark is resized to the available bit capacity automatically, so the portal accepts arbitrary watermark dimensions.
- The watermarked image is cropped back to the original cover size for display and download, while the padded version is used internally for robust block processing.